# Financial Anomaly Detection — Exploratory Analysis

This notebook walks through the full pipeline interactively:
1. Load and explore the synthetic dataset
2. Validate and preprocess the data
3. Run Isolation Forest + statistical baseline
4. Visualise results
5. Evaluate with ground-truth labels

> **Disclaimer:** This notebook uses a synthetic dataset. Anomaly alerts do not confirm fraud.


In [ ]:
import sys
from pathlib import Path

# Make sure project root is on the path
ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded.')

## 1. Load the Sample Dataset

In [ ]:
from src.data_loader import load_csv, validate_dataframe
from config import SAMPLE_DATA_PATH

df_raw = load_csv(SAMPLE_DATA_PATH)
print(f'Loaded {len(df_raw):,} rows, {len(df_raw.columns)} columns')
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe().round(2)

## 2. Validate the Dataset

In [ ]:
val_result = validate_dataframe(df_raw)
print(val_result.summary())

## 3. Label Distribution

In [ ]:
label_counts = df_raw['is_fraud'].value_counts()
print(label_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution by label
for label, group in df_raw.groupby('is_fraud'):
    axes[0].hist(group['amount'], bins=50, alpha=0.6,
                 label=f'{"Anomaly" if label else "Normal"} (n={len(group)})')
axes[0].set_xlabel('Amount (USD)')
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Amount Distribution')
axes[0].legend()
axes[0].set_yscale('log')

# Pie chart
axes[1].pie(label_counts, labels=['Normal', 'Anomaly'],
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'],
            startangle=140)
axes[1].set_title('Normal vs Anomaly')

plt.tight_layout()
plt.show()

## 4. Preprocess

In [ ]:
from src.preprocessing import preprocess_dataframe, get_feature_names

df_processed, X, encoders = preprocess_dataframe(df_raw)
feature_names = get_feature_names(df_processed)

print(f'Processed shape: {df_processed.shape}')
print(f'Feature matrix shape: {X.shape}')
print(f'Features used: {feature_names}')

## 5. Anomaly Detection

In [ ]:
from src.anomaly_detection import run_detection

result_df, ifd, sbd = run_detection(
    df_processed, X, feature_names,
    contamination=0.05,
    score_threshold=-0.1,
    z_threshold=3.0
)

n_flagged = int(result_df['is_anomaly'].sum())
print(f'Flagged: {n_flagged} / {len(result_df)} ({100*n_flagged/len(result_df):.1f}%)')
result_df[result_df['is_anomaly']].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly score distribution
normal_scores = result_df[~result_df['is_anomaly']]['anomaly_score']
flagged_scores = result_df[result_df['is_anomaly']]['anomaly_score']

axes[0].hist(normal_scores, bins=40, alpha=0.7, color='#2196F3', label='Normal')
axes[0].hist(flagged_scores, bins=20, alpha=0.7, color='#F44336', label='Flagged')
axes[0].axvline(-0.1, color='orange', linestyle='--', label='Threshold (-0.1)')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Anomaly Score Distribution')
axes[0].legend()

# Score vs Amount
normal = result_df[~result_df['is_anomaly']]
flagged = result_df[result_df['is_anomaly']]
axes[1].scatter(normal['amount'], normal['anomaly_score'],
                alpha=0.4, color='#2196F3', s=15, label='Normal')
axes[1].scatter(flagged['amount'], flagged['anomaly_score'],
                alpha=0.8, color='#F44336', s=30, label='Flagged')
axes[1].set_xscale('log')
axes[1].axhline(-0.1, color='orange', linestyle='--', label='Threshold')
axes[1].set_xlabel('Amount (USD, log scale)')
axes[1].set_ylabel('Anomaly Score')
axes[1].set_title('Score vs Amount')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Model Evaluation (Supervised)

In [ ]:
from src.evaluation import auto_evaluate

eval_result = auto_evaluate(result_df)
print(eval_result.summary())

In [ ]:
if eval_result.confusion is not None:
    import matplotlib.pyplot as plt
    import numpy as np

    cm = eval_result.confusion
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Normal', 'Anomaly'])
    ax.set_yticklabels(['Normal', 'Anomaly'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix — Isolation Forest')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=16)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

## 7. Sample Flagged Transactions with Explanations

In [ ]:
cols_to_show = ['transaction_id', 'timestamp', 'amount', 'merchant_category',
                'country_code', 'card_present', 'anomaly_score', 'explanation']
cols_to_show = [c for c in cols_to_show if c in result_df.columns]

flagged_df = result_df[result_df['is_anomaly']].sort_values('anomaly_score').head(10)
for _, row in flagged_df.iterrows():
    print(f"\n{'='*70}")
    print(f"ID: {row['transaction_id']}  |  Amount: ${row['amount']:,.2f}  |  Score: {row['anomaly_score']:.4f}")
    print(f"Explanation: {row.get('explanation', 'N/A')}")

## 8. Generate and Preview Report

In [ ]:
from src.reporting import generate_summary_text

summary = generate_summary_text(result_df, eval_result)
print(summary)